# Comparativo: prever depressão com e sem variáveis de ansiedade

## Motivação

O trabalho foi feito em duas etapas, e este notebook reúne as duas em um único experimento reprodutível:

1. **Cenário A – exploratório (com ansiedade):** removemos de $X$ apenas os sintomas diretos de depressão (PHQ) e **mantivemos** as variáveis de ansiedade (GAD-7). O objetivo foi *explorar os dados* e entender quanto a ansiedade, fortemente comórbida com a depressão, ajuda a prever o PHQ.
2. **Cenário B – hipótese do projeto (sem sintomas de saúde mental):** removemos de $X$ **todos** os sintomas de depressão **e** de ansiedade. Sobram apenas variáveis sociodemográficas, de saúde física (doenças e condições crônicas), de acesso a serviços de saúde e de contexto social.

O cenário A serve de **referência (teto de desempenho)**; o cenário B é o que realmente testa a nossa hipótese.

## Hipótese

> É possível prever depressão a partir de dados de doenças físicas e do contexto de vida, **sem** depender de sintomas de transtornos mentais.

É esperado que o cenário B tenha desempenho **pior** que o A, pois a ansiedade carrega muita informação sobre a depressão. A hipótese **não** exige que B iguale A: ela se sustenta se B ainda tiver **poder preditivo real, bem acima do acaso** (ROC-AUC bem acima de 0,5 e recall/precisão úteis para triagem). Por isso, a comparação importa tanto quanto o valor absoluto das métricas.

## Desenho do experimento (para a comparação ser justa)

- Mesma base (`adult25_limpo.csv`), mesmo alvo binário (`0` = sem depressão, `1` = com depressão) e mesmo `PESO_AMOSTRAL`.
- **Mesmas linhas e mesmo split treino/teste** nos dois cenários (registros sem `-1` nas variáveis do cenário A, a base mais restritiva); só muda o conjunto de colunas.
- Mesmo modelo e mesmos hiperparâmetros (Random Forest), com métricas ponderadas pela amostra.

# 1. Preparação dos dados e definição dos cenários

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET_ORIGINAL = 'SEVERIDADE_DEPRESSAO_PHQ'
TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
RANDOM_STATE = 42

# Sintomas diretos de depressão (PHQ): removidos nos DOIS cenários
COLUNAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]

# Ansiedade (GAD-7): mantida no cenário A, removida no cenário B
COLUNAS_ANSIEDADE = [
    'HISTORICO_TRANSTORNO_ANSIEDADE', 'MED_ANSIEDADE',
    'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE',
    'FREQ_ANSIEDADE_2SEM', 'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
    'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
    'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM',
    'FREQ_MEDO_2SEM', 'SEVERIDADE_ANSIEDADE'
]

# Carregar a base limpa
df = pd.read_csv(Path('../../01 Preparação dos dados/dados/limpos') / 'adult25_limpo.csv').set_index('ID_DOMICILIO')

# Garante que nenhuma coluna foi digitada errado (o drop abaixo ignoraria erros silenciosamente)
ausentes = [c for c in COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE if c not in df.columns]
assert not ausentes, f'Colunas inexistentes na base: {ausentes}'

# Remover registros indefinidos no alvo (-1 e 5) e binarizar: 0 = sem depressão | 1 = com depressão
df = df[~df[TARGET_ORIGINAL].isin([-1, 5])].copy()
df[TARGET_BIN] = (df[TARGET_ORIGINAL] != 1).astype(int)

# Definição dos dois cenários: o que é removido de X em cada um
CENARIOS = {
    'A: com ansiedade (exploratório)': COLUNAS_DEPRESSAO,
    'B: sem sintomas mentais (hipótese)': COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE,
}
BASE_REMOVER = [TARGET_BIN, PESO_COLUNA]

# Mesmas linhas nos dois cenários: registros sem -1 nas variáveis do cenário A (o mais restritivo)
X_cenario_a = df.drop(columns=BASE_REMOVER + CENARIOS['A: com ansiedade (exploratório)'])
mascara_completos = (X_cenario_a != -1).all(axis=1).to_numpy()
df_modelo = df[mascara_completos].copy()

# Mesmo split nos dois cenários (por posição)
posicoes = np.arange(len(df_modelo))
pos_tr, pos_te = train_test_split(
    posicoes, test_size=0.2, random_state=RANDOM_STATE, stratify=df_modelo[TARGET_BIN]
)

y_tr = df_modelo[TARGET_BIN].iloc[pos_tr]
y_te = df_modelo[TARGET_BIN].iloc[pos_te]
w_tr = df_modelo[PESO_COLUNA].iloc[pos_tr]
w_te = df_modelo[PESO_COLUNA].iloc[pos_te]

print(f'Registros no experimento: {len(df_modelo)} (treino: {len(pos_tr)} | teste: {len(pos_te)})')
print(f'Proporção de depressão (ponderada pelo peso amostral): {np.average(df_modelo[TARGET_BIN], weights=df_modelo[PESO_COLUNA]):.1%}')
for nome, remover in CENARIOS.items():
    n_feat = df_modelo.drop(columns=BASE_REMOVER + remover).shape[1]
    print(f'Cenário {nome}: {n_feat} variáveis preditoras')

Registros no experimento: 22084 (treino: 17667 | teste: 4417)
Proporção de depressão (ponderada pelo peso amostral): 21.2%
Cenário A: com ansiedade (exploratório): 58 variáveis preditoras
Cenário B: sem sintomas mentais (hipótese): 46 variáveis preditoras


# 2. Função de treino e avaliação

A mesma função é usada nos dois cenários, garantindo que a **única diferença** entre eles seja o conjunto de variáveis. Retorna as métricas da classe positiva (com depressão), ponderadas pelo `PESO_AMOSTRAL`, a matriz de confusão em **contagem de registros reais** do teste (sem peso), a matriz ponderada (usada só na análise final) e a importância das variáveis do modelo.

In [8]:
def treinar_avaliar(colunas_remover):
    X = df_modelo.drop(columns=BASE_REMOVER + colunas_remover)
    X_tr, X_te = X.iloc[pos_tr], X.iloc[pos_te]

    num_cols = [c for c in ['RAZAO_RENDA_POBREZA'] if c in X.columns]
    cat_cols = [c for c in X.columns if c not in num_cols]
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ])
    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_te_proc = preprocessor.transform(X_te)

    modelo = RandomForestClassifier(
        n_estimators=300, max_depth=25, min_samples_split=5,
        max_features='log2', class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    modelo.fit(X_tr_proc, y_tr, sample_weight=w_tr)

    y_pred = modelo.predict(X_te_proc)
    y_proba = modelo.predict_proba(X_te_proc)[:, 1]

    metricas = {
        'Nº de variáveis': X.shape[1],
        'ROC-AUC': roc_auc_score(y_te, y_proba, sample_weight=w_te),
        'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
        'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    }
    matriz = confusion_matrix(y_te, y_pred)
    matriz_pond = np.round(confusion_matrix(y_te, y_pred, sample_weight=w_te)).astype(int)

    # Importância agregada por variável original (soma das colunas One-Hot de cada uma)
    nomes = preprocessor.get_feature_names_out()
    importancias = pd.Series(modelo.feature_importances_, index=nomes)
    originais = sorted(X.columns, key=len, reverse=True)
    agregada = {}
    for nome_col, imp in importancias.items():
        base = nome_col.split('__', 1)[1]
        dono = next((c for c in originais if base == c or base.startswith(c + '_')), base)
        agregada[dono] = agregada.get(dono, 0) + imp
    importancia_var = pd.Series(agregada).sort_values(ascending=False)

    return metricas, matriz, matriz_pond, importancia_var

resultados = {}

# 3. Cenário A: com variáveis de ansiedade (exploratório)

Removemos de $X$ somente os sintomas de depressão (PHQ). As variáveis de ansiedade (GAD-7) permanecem. Como ansiedade e depressão são fortemente comórbidas, este cenário mostra o **melhor desempenho que conseguimos quando há informação de saúde mental disponível**.

In [9]:
nome_a = 'A: com ansiedade (exploratório)'
resultados[nome_a] = treinar_avaliar(CENARIOS[nome_a])
metricas_a, matriz_a, matriz_pond_a, imp_a = resultados[nome_a]

print(f'--- {nome_a} ---')
for k, v in metricas_a.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_a)
print('\nTop 10 variáveis mais importantes:')
print(imp_a.head(10).round(4).to_string())

--- A: com ansiedade (exploratório) ---
Nº de variáveis: 58
ROC-AUC: 0.9135
Acurácia: 0.8606
Precisão (depressão): 0.6556
Recall (depressão): 0.7491
F1 (depressão): 0.6993

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3113  369]
 [ 229  706]]

Top 10 variáveis mais importantes:
TIPO_DIFICULDADE_COGNITIVA             0.0607
SEVERIDADE_ANSIEDADE                   0.0598
FREQ_ANSIEDADE                         0.0571
IDADE                                  0.0516
FREQ_ANSIEDADE_2SEM                    0.0469
FREQ_PREOCUPACAO_DESNECESSARIA_2SEM    0.0458
FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM    0.0447
FREQ_DIFICULDADE_DESCANSAR_2SEM        0.0399
FREQ_IRRITABILIDADE_2SEM               0.0339
FREQ_MEDO_2SEM                         0.0302


# 4. Cenário B: sem sintomas de saúde mental (hipótese do projeto)

Removemos de $X$ os sintomas de depressão **e** de ansiedade. O modelo passa a usar apenas variáveis sociodemográficas, de saúde física (doenças e condições crônicas), de acesso a serviços de saúde e de contexto social. É aqui que testamos se dá para **prever depressão a partir de doenças físicas e do contexto de vida**.

In [10]:
nome_b = 'B: sem sintomas mentais (hipótese)'
resultados[nome_b] = treinar_avaliar(CENARIOS[nome_b])
metricas_b, matriz_b, matriz_pond_b, imp_b = resultados[nome_b]

print(f'--- {nome_b} ---')
for k, v in metricas_b.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_b)
print('\nTop 10 variáveis mais importantes:')
print(imp_b.head(10).round(4).to_string())

--- B: sem sintomas mentais (hipótese) ---
Nº de variáveis: 46
ROC-AUC: 0.8411
Acurácia: 0.8153
Precisão (depressão): 0.5713
Recall (depressão): 0.5857
F1 (depressão): 0.5784

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3017  465]
 [ 350  585]]

Top 10 variáveis mais importantes:
IDADE                          0.0937
TIPO_DIFICULDADE_COGNITIVA     0.0920
FREQ_SOLITARIO                 0.0716
SUPORTE_SOCIAL                 0.0579
QTD_DIFICULDADE_COGNITIVA      0.0476
AUTOAVALIACAO_SAUDE_GERAL      0.0457
FREQUENCIA_FALHA_MEMORIA       0.0452
CATEGORIA_RAZAO_POBREZA        0.0436
DOR_CRONICA                    0.0409
ESCOLARIDADE_MAXIMA_FAMILIA    0.0346


# 5. Comparativo entre os cenários

A coluna **Variação (B − A)** mostra quanto desempenho se perde ao retirar a ansiedade. A coluna **Referência (acaso)** mostra o que um modelo sem nenhuma informação conseguiria: ROC-AUC de 0,5 e precisão igual à prevalência de depressão.

In [11]:
prevalencia = np.average(y_te, weights=w_te)

tabela = pd.DataFrame({nome: res[0] for nome, res in resultados.items()})
tabela['Variação (B − A)'] = tabela[nome_b] - tabela[nome_a]
tabela['Referência (acaso)'] = [np.nan, 0.5, np.nan, prevalencia, np.nan, np.nan]

print(f'Prevalência de depressão no teste (ponderada): {prevalencia:.1%}\n')
tabela.round(4)

Prevalência de depressão no teste (ponderada): 21.6%



,A: com ansiedade (exploratório),B: sem sintomas mentais (hipótese),Variação (B − A),Referência (acaso)
Nº de variáveis,58.0000,46.0000,-12.0000,NaN
ROC-AUC,0.9135,0.8411,-0.0724,0.5000
Acurácia,0.8606,0.8153,-0.0453,NaN
Precisão (depressão),0.6556,0.5713,-0.0843,0.2163
Recall (depressão),0.7491,0.5857,-0.1635,NaN
F1 (depressão),0.6993,0.5784,-0.1209,NaN


In [12]:
# Variáveis que o cenário B (sem sintomas mentais) usa para prever depressão
print('Top 15 variáveis do cenário B (importância agregada por variável original):\n')
print(imp_b.head(15).round(4).to_string())

ganho_auc = metricas_b['ROC-AUC'] - 0.5
perda_auc = metricas_a['ROC-AUC'] - metricas_b['ROC-AUC']
print(f"\nROC-AUC do cenário B acima do acaso: +{ganho_auc:.3f}")
print(f"ROC-AUC perdido ao retirar a ansiedade (A → B): -{perda_auc:.3f}")

# Matrizes ponderadas pelo PESO_AMOSTRAL: projeção para a população, útil só para a análise final
print('\nMatrizes de confusão ponderadas (projeção populacional; linhas = real, colunas = previsto):')
for nome, mp in [(nome_a, matriz_pond_a), (nome_b, matriz_pond_b)]:
    print(f'\n{nome}')
    print(mp)

Top 15 variáveis do cenário B (importância agregada por variável original):

IDADE                           0.0937
TIPO_DIFICULDADE_COGNITIVA      0.0920
FREQ_SOLITARIO                  0.0716
SUPORTE_SOCIAL                  0.0579
QTD_DIFICULDADE_COGNITIVA       0.0476
AUTOAVALIACAO_SAUDE_GERAL       0.0457
FREQUENCIA_FALHA_MEMORIA        0.0452
CATEGORIA_RAZAO_POBREZA         0.0436
DOR_CRONICA                     0.0409
ESCOLARIDADE_MAXIMA_FAMILIA     0.0346
RAZAO_RENDA_POBREZA             0.0338
TEMPO_MORADIA_ANOS              0.0240
LIMITACAO_TRABALHO_POR_SAUDE    0.0225
ESTRESSE_FINANCEIRO_MEDICO      0.0224
CATEGORIA_IMC                   0.0215

ROC-AUC do cenário B acima do acaso: +0.341
ROC-AUC perdido ao retirar a ansiedade (A → B): -0.072

Matrizes de confusão ponderadas (projeção populacional; linhas = real, colunas = previsto):

A: com ansiedade (exploratório)
[[33671240  4101965]
 [ 2615251  7810026]]

B: sem sintomas mentais (hipótese)
[[33192095  4581109]
 [ 4319371  

# 6. Interpretação e conclusão

**O que esperávamos e por quê.** O cenário A usa a ansiedade, que é um dos preditores mais fortes de depressão (comorbidade). Ao retirá-la no cenário B, **é natural que as métricas piorem**, e essa queda é o custo de não depender de sintomas de saúde mental. Isso não invalida a hipótese.

**O que a hipótese afirma.** A hipótese do projeto não é que o modelo sem ansiedade seja *tão bom quanto* o modelo com ansiedade, e sim que **é possível prever depressão usando dados de doenças físicas e do contexto de vida**. Ela se sustenta quando o cenário B mostra capacidade preditiva **claramente acima do acaso**, e isso se verifica comparando:

- o **ROC-AUC** do cenário B com a referência de 0,5 (modelo sem informação);
- a **precisão** do cenário B com a prevalência de depressão na população (taxa de acerto de um palpite aleatório);
- o **recall**, que indica quantos casos de depressão o modelo ainda consegue encontrar sem ver nenhum sintoma mental.

**Por que isso importa.** Um modelo que depende de sintomas de saúde mental só funciona depois que a pessoa já respondeu a questionários como o PHQ ou o GAD-7, ou seja, quando o quadro já foi observado. Um modelo baseado em condições físicas e em fatores sociais poderia ser usado como **triagem populacional**, sinalizando grupos de maior risco a partir de dados que já existem em prontuários e pesquisas de saúde.

**Resultado da execução (`random_state=42`, 22.084 registros, prevalência de depressão de ~21,6% no teste).**

| Métrica | A: com ansiedade | B: sem sintomas mentais | Acaso |
|---|---|---|---|
| ROC-AUC | 0,914 | 0,841 | 0,500 |
| Precisão (depressão) | 65,6% | 57,1% | 21,6% |
| Recall (depressão) | 74,9% | 58,6% | – |
| F1 (depressão) | 0,699 | 0,578 | – |

- Como esperado, **B é pior que A** (queda de 0,072 no ROC-AUC e de 16 p.p. no recall).
- Ainda assim, **B sustenta a hipótese**: o ROC-AUC é 0,34 acima do acaso e a precisão (57%) é mais de **2,6 vezes** a prevalência de depressão (22%). Sem ver nenhum sintoma de saúde mental, o modelo ainda encontra cerca de 59% dos casos.
- **Atenção ao que sustenta o cenário B.** As variáveis mais importantes são idade, dificuldade cognitiva, solidão (`FREQ_SOLITARIO`), suporte social e autoavaliação de saúde, com dor crônica e limitação de trabalho por saúde logo depois. Ou seja, o sinal vem de uma **combinação de saúde física/funcional e contexto social**, e não apenas de doenças físicas isoladas. Para defender a hipótese "doenças físicas" em específico, vale um cenário C que use só variáveis de saúde física, para medir a contribuição delas sozinhas.

Os números acima vêm de uma execução do notebook. Rode as células para reproduzi-los e compare sempre a linha de ROC-AUC e a de precisão do cenário B com a coluna *Referência (acaso)* da tabela comparativa.

**Limitações.**

- O desempenho vem de **uma única partição de teste**; diferenças pequenas devem ser lidas com cautela.
- O modelo identifica **associações**, não causas: doenças físicas e depressão podem se influenciar mutuamente, e fatores sociais podem confundir o resultado.
- O alvo (PHQ) é um instrumento de **rastreio**, e não um diagnóstico clínico.
- Os hiperparâmetros foram mantidos iguais nos dois cenários para a comparação ser justa; o cenário B pode se beneficiar de uma busca própria de hiperparâmetros.